In [2]:
import os
import csv
import hashlib
import uuid
import re
import time
import random
import logging
from datetime import datetime
from duckduckgo_search import DDGS
from collections import defaultdict

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

# ============================================================
# ⚙️  CONFIG
# ============================================================
CONFIG = {
    "csv_filename": "jobs_extracted_data.csv", # మీ CSV ఫైల్ పేరు ఇక్కడ మార్చుకోవచ్చు
    "max_results": 15,
    "time_filter": "d",       # 'd' = last 24 hours (yesterday + today)
    "min_score": 35,          # Realistic threshold for job boards
    # తప్పకుండా! మీరు అడిగినట్లుగా PySpark/Delta Table కి సంబంధించిన కోడ్‌ని తీసేసి, దాని ప్లేస్‌లో డేటాని డైరెక్ట్ గా ఒక **CSV ఫైల్** లో సేవ్ చేసేలా కోడ్ మార్చాను. 
}

In [3]:

import hashlib
import uuid
import re
import time
import random
import logging
import csv
from datetime import date, datetime
from collections import Counter
from duckduckgo_search import DDGS

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

# ============================================================
# ⚙️  CONFIG
# ============================================================
CONFIG = {
    "csv_file": "jobs_automation_output.csv", # మీ CSV ఫైల్ పేరు ఇక్కడ మార్చుకోవచ్చు
    "max_results": 15,
    "time_filter": "d",       # 'd' = last 24 hours (yesterday + today)
    "min_score": 35,          # Realistic threshold for job boards
    "search_delay": (2.0, 4.0),
}

# ============================================================
# 🎯 TARGET PORTALS
# ============================================================
PORTAL_SITES = [
    "site:[linkedin.com/jobs](https://linkedin.com/jobs)",
    "site:indeed.com",
    "site:dice.com",
    "site:glassdoor.com",
    "site:wellfound.com",
    "site:builtin.com",
    "site:ziprecruiter.com",
]

SNIPPET_TRUSTED_SITES = {
    "linkedin.com", "indeed.com", "glassdoor.com", 
    "ziprecruiter.com", "dice.com", "wellfound.com", "builtin.com"
}

ROLES = [
    "Data Engineer", "Senior Data Engineer",
    "PySpark Engineer", "Spark Developer",
    "ETL Developer", "Data Pipeline Engineer",
    "Analytics Engineer", "BI Engineer",
    "Python Developer", "Python Engineer",
    "Machine Learning Engineer", "MLOps Engineer"
]

GARBAGE_DOMAINS = {"youtube.com", "github.com", "stackoverflow.com", "medium.com", "reddit.com", "quora.com", "udemy.com", "coursera.org"}

# ============================================================
# SECTION 1: SMART VALIDATORS
# ============================================================
def is_garbage_url(url: str) -> bool:
    return any(d in url.lower() for d in GARBAGE_DOMAINS)

def is_real_job_page(text: str) -> bool:
    if not text or len(text) < 80:
        return False
    low = text.lower()
    job_keywords = ["experience", "skills", "requirements", "responsibilities", "role", "candidate", "apply", "qualifications", "hiring"]
    return sum(1 for kw in job_keywords if kw in low) >= 2

def compute_validation_score(rec: dict) -> int:
    score = 0
    desc  = rec.get("job_description", "").lower()
    title = rec.get("job_title", "").lower()
    company = rec.get("company_name", "").lower()

    if len(title) > 5 and "unknown" not in title and "job" not in title:
        score += 30
    if len(company) > 3 and "unknown" not in company and "web extracted" not in company:
        score += 20

    kw_count = sum(1 for kw in ["experience", "skills", "requirements", "responsibilities"] if kw in desc)
    if kw_count >= 2: score += 30
    elif kw_count >= 1: score += 15

    if len(desc) > 300: score += 20
    elif len(desc) > 100: score += 10

    return min(score, 100)

# ============================================================
# SECTION 2: TEXT CLEANERS
# ============================================================
def clean_title(raw: str) -> str:
    s = re.sub(r"\[.*?\]|\(.*?\)", "", raw)
    s = re.sub(r"(?i)(Job Application for|Apply for|Jobs In)\s+", "", s)
    for suffix in [" - LinkedIn", " | LinkedIn", " - Indeed", " | Glassdoor", " - ZipRecruiter", " | Wellfound"]:
        s = re.sub(rf"(?i){suffix}.*$", "", s)
    return s.strip()

def split_title_and_company(raw_title: str):
    cleaned = clean_title(raw_title)
    job_title, company = cleaned, "Unknown"
    for sep in [" at ", " At ", " - ", " | ", " @ "]:
        if sep in cleaned:
            parts = cleaned.split(sep, 1)
            job_title, company = parts[0].strip(), parts[1].strip()
            break
    
    job_title = re.sub(r"[^a-zA-Z0-9\s\+\#\.]", "", job_title).strip().title()
    company = re.sub(r"[^a-zA-Z0-9\s\.,&\-]", "", company).strip().title()
    return (job_title or "Unknown"), (company or "Unknown")

# ============================================================
# SECTION 3: RESULT PROCESSOR
# ============================================================
def process_result(res: dict, canonical_role: str) -> dict | None:
    url = (res.get("href") or "").strip()
    snippet = (res.get("body") or "").strip()
    s_title = (res.get("title") or "").strip()

    if not url or is_garbage_url(url):
        return None

    page_title = s_title
    description = snippet

    if not is_real_job_page(description):
        return None

    job_title, company = split_title_and_company(page_title)
    if len(job_title) < 3:
        job_title = canonical_role.title()

    now = datetime.now()
    rec = {
        "id":                  str(uuid.uuid4()),
        "job_hash":            hashlib.md5(url.encode()).hexdigest(),
        "created_at":          now.strftime("%Y-%m-%d %H:%M:%S"),
        "updated_at":          now.strftime("%Y-%m-%d %H:%M:%S"),
        "fetch_date":          now.strftime("%Y-%m-%d"),
        "search_keyword":      canonical_role,
        "company_name":        company,
        "job_title":           job_title,
        "job_description":     description,
        "apply_link":          url,
        "hr_email":            "Not Found",
        "job_type":            "Remote" if "remote" in description.lower() else "Not Specified",
        "salary_range":        "Not Specified",
        "experience_required": "Not Specified",
        "location":            "USA",
        "skills_required":     "Not Specified",
        "link_status":         "Active",
        "validation_score":    0,
        "validation_status":   "Pending",
    }
    
    score = compute_validation_score(rec)
    rec["validation_score"] = score
    rec["validation_status"] = "Valid" if score >= 70 else "Partial" if score >= CONFIG["min_score"] else "Junk"

    if score < CONFIG["min_score"]:
        return None

    return rec

# ============================================================
# SECTION 4: SEARCH ENGINE
# ============================================================
def search_for_role(canonical_role: str) -> list:
    records = []
    seen_urls = set()
    phrase = f'"{canonical_role}"'
    
    for portal in PORTAL_SITES:
        query = f"{phrase} {portal}"
        log.info(f"🔍 Searching: {query}")
        
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(
                    query,
                    region="us-en",
                    max_results=CONFIG["max_results"],
                    timelimit=CONFIG["time_filter"]
                ))
            
            if not results:
                continue
                
            log.info(f"  📥 Found {len(results)} links. Processing...")
            
            for res in results:
                href = res.get("href", "")
                if href in seen_urls:
                    continue
                seen_urls.add(href)
                
                job = process_result(res, canonical_role)
                if job:
                    records.append(job)
                    
            time.sleep(random.uniform(*CONFIG["search_delay"]))
            
        except Exception as e:
            log.warning(f"  ⚠️ Search failed for {portal}: {str(e)[:50]}")
            time.sleep(3)

    log.info(f"  ✅ {canonical_role} → {len(records)} valid jobs found")
    return records

# ============================================================
# SECTION 5: CSV WRITER
# ============================================================
def write_to_csv(records: list) -> int:
    if not records:
        return 0
    
    csv_file = CONFIG["csv_file"]
    
    # Get headers from the first dictionary keys
    headers = list(records[0].keys())
    
    # Writing to CSV
    try:
        with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=headers)
            writer.writeheader()
            writer.writerows(records)
        return len(records)
    except Exception as e:
        log.error(f"❌ Failed to write to CSV: {e}")
        return 0

# ============================================================
# ▶️  MAIN ORCHESTRATOR
# ============================================================
def run_harvester_v8():
    log.info("=" * 65)
    log.info("🚀 US IT JOB HARVESTER V8 — SMART SNIPPET STRATEGY")
    log.info("🎯 Target: LinkedIn, Indeed, Dice, Glassdoor, Wellfound, Built In, ZipRecruiter")
    log.info("⏱️  Filter: Last 24 Hours (Yesterday + Today)")
    log.info("=" * 65)

    all_records = []
    for role in ROLES:
        try:
            jobs = search_for_role(role)
            all_records.extend(jobs)
        except Exception as e:
            log.error(f"❌ Error for {role}: {e}")
        time.sleep(random.uniform(2.0, 4.0))

    # Global Dedup
    seen, unique = set(), []
    for rec in all_records:
        if rec["job_hash"] not in seen:
            seen.add(rec["job_hash"])
            unique.append(rec)

    log.info(f"\n📊 Total Raw: {len(all_records)} | After Dedup: {len(unique)}")
    
    # Write to CSV instead of Delta Table
    written = write_to_csv(unique)
    log.info(f"🎉 HARVEST COMPLETE — {written} new records written to {CONFIG['csv_file']}!")

    # Simple Summary without Spark SQL
    if unique:
        print("\n" + "="*40)
        print("📊 SUMMARY (Role & Status Count)")
        print("="*40)
        summary = Counter((r["search_keyword"], r["validation_status"]) for r in unique)
        for (role, status), count in summary.most_common():
            print(f"🔹 {role:<25} | {status:<7} | {count} jobs")
        print("="*40 + "\n")

# 🚀 RUN IT
if __name__ == "__main__":
    run_harvester_v8()

18:05:42 [INFO] =================================================================
18:05:42 [INFO] 🚀 US IT JOB HARVESTER V8 — SMART SNIPPET STRATEGY
18:05:42 [INFO] 🎯 Target: LinkedIn, Indeed, Dice, Glassdoor, Wellfound, Built In, ZipRecruiter
18:05:42 [INFO] ⏱️  Filter: Last 24 Hours (Yesterday + Today)
18:05:42 [INFO] =================================================================
18:05:42 [INFO] 🔍 Searching: "Data Engineer" site:[linkedin.com/jobs](https://linkedin.com/jobs)
/var/folders/lv/sb2c9b1508501k5dv4pxz4740000gn/T/ipykernel_21671/2788779757.py:177: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
18:05:43 [INFO] response: https://www.bing.com/search?q=%22Data+Engineer%22+site%3A%5Blinkedin.com%2Fjobs%5D%28https%3A%2F%2Flinkedin.com%2Fjobs%29&filters=ex1%3A%22ez1%22 200
18:05:43 [INFO] 🔍 Searching: "Data Engineer" site:indeed.com
/var/folders/lv/sb2c9b1508501k5dv4pxz4740000gn/T/ipykernel_21


📊 SUMMARY (Role & Status Count)
🔹 Senior Data Engineer      | Partial | 1 jobs
🔹 Data Pipeline Engineer    | Partial | 1 jobs

